<a href="https://colab.research.google.com/github/Edenshmuel/CrimeData/blob/main/DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Required Libraries

In [1]:
import zipfile
import requests
from io import BytesIO
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

Loading Supervised Datasets from GitHub Repository

In [2]:
# Define the ZIP file URL
zip_url = "https://github.com/Edenshmuel/CrimeData/raw/main/supervised_dataset.zip"

# Function to load a specific CSV file from the ZIP in GitHub
def load_csv_from_zip(zip_url, inner_file_name):
    response = requests.get(zip_url)
    if response.status_code == 200:
        with zipfile.ZipFile(BytesIO(response.content)) as z:
            with z.open(inner_file_name) as f:
                return pd.read_csv(f)
    else:
        raise Exception("Failed to download supervised_dataset.zip")

# Load datasets from the ZIP
X_train = load_csv_from_zip(zip_url, "X_train_supervised.csv")
X_test = load_csv_from_zip(zip_url, "X_test_supervised.csv")
y_train = load_csv_from_zip(zip_url, "y_train_supervised.csv").values.ravel()
y_test = load_csv_from_zip(zip_url, "y_test_supervised.csv").values.ravel()

Building and Training a Neural Network with Early Stopping and Learning Rate Reduction

In [4]:
# Define early stopping and learning rate reduction callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Build the neural network model
model = Sequential()

# Input layer with L2 regularization, LeakyReLU activation, and dropout
model.add(Dense(512, kernel_regularizer=l2(0.01), input_shape=(X_train.shape[1],)))
model.add(LeakyReLU(alpha=0.01))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Hidden layer 1
model.add(Dense(256, kernel_regularizer=l2(0.01)))
model.add(LeakyReLU(alpha=0.01))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Hidden layer 2
model.add(Dense(128, kernel_regularizer=l2(0.01)))
model.add(LeakyReLU(alpha=0.01))
model.add(BatchNormalization())
model.add(Dropout(0.3))

# Output layer with softmax activation for multi-class classification
model.add(Dense(len(np.unique(y_train)), activation='softmax'))

# Compile the model with Adam optimizer and sparse categorical cross-entropy
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model with early stopping and learning rate reduction
history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=512,
                    validation_split=0.2,
                    callbacks=[early_stopping, reduce_lr])

Epoch 1/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - accuracy: 0.3180 - loss: 2.7453 - val_accuracy: 0.3879 - val_loss: 1.5894 - learning_rate: 0.0010
Epoch 2/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.3590 - loss: 1.6172 - val_accuracy: 0.3360 - val_loss: 1.5939 - learning_rate: 0.0010
Epoch 3/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.3612 - loss: 1.6101 - val_accuracy: 0.3257 - val_loss: 1.5907 - learning_rate: 0.0010
Epoch 4/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.3624 - loss: 1.6078 - val_accuracy: 0.3876 - val_loss: 1.5771 - learning_rate: 0.0010
Epoch 5/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.3617 - loss: 1.6037 - val_accuracy: 0.3872 - val_loss: 1.7684 - learning_rate: 0.0010
Epoch 6/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.3649 - loss: 1.6006 - val_accuracy: 0.3872 - val_loss: 1.6391 - learning_rate: 0.0010
Epoch 7/50
2129/2129 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.3724 -

Evaluating Model Performance on the Test Set

In [6]:
# Evaluate the model on the test set and print the accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

12679/12679 ━━━━━━━━━━━━━━━━━━━━ 35s 3ms/step - accuracy: 0.3901 - loss: 1.5546
Test Accuracy: 39.11%


Generating Predictions Using the Trained Model

In [7]:
# Predict class probabilities and convert them to class labels
y_pred_dnn = model.predict(X_test)
y_pred_dnn_classes = np.argmax(y_pred_dnn, axis=1)

12679/12679 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step


Evaluation Function for Classification Models

In [8]:
# Define a function to calculate and return multiple evaluation metrics
def evaluate_model(y_true, y_pred, average='weighted'):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, average=average, zero_division=0),
        "Confusion Matrix": confusion_matrix(y_true, y_pred)
    }
    return metrics

Evaluating the DNN Model on the Test Set

In [10]:
dnn_metrics = evaluate_model(y_test, y_pred_dnn_classes)

for metric, value in dnn_metrics.items():
    print(f"DNN {metric}: {value}")

DNN Accuracy: 0.3910594696120068
DNN Precision: 0.27963651911484977
DNN Recall: 0.3910594696120068
DNN F1 Score: 0.31620844842674817
DNN Confusion Matrix: [[    0     0     0     0   240     0     0     0     0     0   198     0
      0     0]
 [    0   222     0     0   859    11     0     0     0     0  6057     0
      0     0]
 [    0     1     0     0  2353     0     0     0     0     0  1899     0
      0     0]
 [    0     5     0     0 11894     7     0     0     0     0 11693     0
      0     0]
 [    0     2     0     0 86910    24     0     0     0     0 52391     0
      0     0]
 [    0    71     0     0  1744    48     0     0     0     0  5893     0
      0     0]
 [    0     0     0     0    63     0     0     0     0     0    41     0
      0     0]
 [    0     2     0     0 10882     5     0     0     0     0  8914     0
      0     0]
 [    0     0     0     0   271     0     0     0     0     0   363     0
      0     0]
 [    0    69     0     0 30871    33     0 

Model confidence analysis

In [11]:
# Compute prediction confidence
confidence = np.max(y_pred_dnn, axis=1)

confidence_df = pd.DataFrame({
    "confidence": confidence,
    "correct": (y_pred_dnn_classes == y_test)
})

print(confidence_df.groupby("correct")["confidence"].mean())

correct
False    0.379609
True     0.388901
Name: confidence, dtype: float32


Accuracy vs confidence threshold

In [12]:
# Evaluate accuracy for high-confidence predictions
threshold = 0.7

high_conf_idx = confidence >= threshold

high_conf_accuracy = np.mean(
    y_pred_dnn_classes[high_conf_idx] == y_test[high_conf_idx]
)

print("Accuracy for high-confidence predictions:", high_conf_accuracy)
print("Fraction of predictions kept:", np.mean(high_conf_idx))

Accuracy for high-confidence predictions: nan
Fraction of predictions kept: 0.0


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Identify the hardest classes for the model

In [14]:
# Per-class accuracy
cm = confusion_matrix(y_test, y_pred_dnn_classes)

per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
per_class_accuracy = np.nan_to_num(per_class_accuracy)

hard_classes = pd.DataFrame({
    "class": np.arange(len(per_class_accuracy)),
    "accuracy": per_class_accuracy
}).sort_values("accuracy")

print("Hardest classes for the model:")
print(hard_classes.head(10))

Hardest classes for the model:
    class  accuracy
0       0       0.0
2       2       0.0
3       3       0.0
7       7       0.0
6       6       0.0
11     11       0.0
9       9       0.0
8       8       0.0
13     13       0.0
12     12       0.0
